# JESUS Film — AI Dubbing PoC (Colab GPU runner)

Produces a **real dubbed + lip-synced clip** in a new language.

**Runtime → Change runtime type → GPU** before you start.

Steps: install → get the code → upload a clip + a ~6s voice reference → run → download.

In [ ]:
#@title 1. Install dependencies (~3-5 min)
!apt-get -qq install -y ffmpeg rubberband-cli > /dev/null
!pip -q install faster-whisper transformers sentencepiece TTS librosa soundfile pyrubberband
import torch; print('GPU available:', torch.cuda.is_available())

In [ ]:
#@title 2. Get the code + Wav2Lip
REPO_URL = 'https://github.com/YOUR_USERNAME/jesus-dub-poc'  #@param {type:'string'}
!git clone -q $REPO_URL || echo 'edit REPO_URL above, or upload the src/ folder manually'
%cd jesus-dub-poc
# Wav2Lip for stage 6 (lip-sync)
!git clone -q https://github.com/Rudrabha/Wav2Lip ../Wav2Lip
print('Download wav2lip_gan.pth into ../Wav2Lip/checkpoints/ (see Wav2Lip README).')

In [ ]:
#@title 3. Sanity check — run the offline demo (proves the pipeline + QA)
!python -m src.pipeline --demo

In [ ]:
#@title 4. Upload your clip (.mp4) and a ~6s voice reference (.wav)
from google.colab import files
import os; os.makedirs('data/input', exist_ok=True)
print('Upload the source clip (mp4):');  up = files.upload()
clip = next(iter(up)); os.replace(clip, 'data/input/clip.mp4')
print('Upload a ~6s reference voice (wav):'); up = files.upload()
ref = next(iter(up)); os.replace(ref, 'data/input/voice_ref.wav')

In [ ]:
#@title 5. Run the full dub (English -> Swahili by default)
SRC = 'eng_Latn' #@param {type:'string'}
TGT = 'swh_Latn' #@param {type:'string'}
XTTS = 'sw'      #@param {type:'string'}
!python -m src.pipeline --input data/input/clip.mp4 --speaker data/input/voice_ref.wav \
    --src-lang $SRC --tgt-lang $TGT --xtts-lang $XTTS --wav2lip-dir ../Wav2Lip

In [ ]:
#@title 6. Preview + download the dubbed clip
from IPython.display import HTML
from base64 import b64encode
mp4 = open('outputs/clip_dubbed.mp4','rb').read()
HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')